# Interactions API의 `input` 형태

`input`에 전달할 수 있는 문자열, Content 배열, Step 배열의 차이를 알아봅니다.

In [3]:
import os

from dotenv import load_dotenv
from google import genai

load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise ValueError(".env 파일에 GEMINI_API_KEY를 설정해 주세요.")

model = os.getenv("GEMINI_MODEL", "gemini-3.6-flash")
client = genai.Client(api_key=api_key)

## 세 가지 형태 한눈에 보기

| 형태 | 용도 | 주요 `type` |
|---|---|---|
| 문자열 | 간단한 텍스트 요청 | 없음 |
| Content 배열 | 한 번의 요청에 여러 종류의 내용 전달 | `text`, `image`, `audio`, `document` |
| Step 배열 | 대화 이력을 직접 구성 | `user_input`, `model_output` 등 |

`user_input`은 대화의 **단계(Step)**이고, `text`나 `image`는 단계 안에 들어가는 **내용(Content)**입니다.

## 1. 문자열

텍스트 하나만 보낼 때 사용하는 가장 간단한 축약형입니다. API가 문자열을 텍스트 Content가 담긴 `user_input` Step으로 변환합니다.

In [4]:
string_response = client.interactions.create(
    model=model,
    input="LLM을 한 문장으로 설명해 줘.",
    store=False,
)

print(string_response.output_text)

**LLM(거대 언어 모델)은 방대한 양의 텍스트 데이터를 학습하여 인간처럼 자연스럽게 언어를 이해하고 글을 작성하는 인공지능입니다.**


## 2. Content 배열

한 번의 사용자 요청에 여러 내용을 함께 넣을 때 사용합니다. 아래 배열 전체는 하나의 `user_input` Step으로 처리됩니다. 멀티모달 요청에서는 같은 배열에 `image`, `audio`, `document` Content를 추가할 수 있습니다.

In [5]:
content_response = client.interactions.create(
    model=model,
    input=[
        {"type": "text", "text": "다음 단어를 한 문장으로 묶어 줘."},
        {"type": "text", "text": "사과, 바나나, 과일 바구니"},
    ],
    store=False,
)

print(content_response.output_text)

제시해주신 단어로 만든 문장입니다.

> "**과일 바구니**에 신선한 **사과**와 **바나나**가 가득 담겨 있습니다."

---

**다른 느낌의 문장들도 추천해 드려요:**
* **행동 중심:** 나는 **사과**와 **바나나**를 예쁜 **과일 바구니**에 정성껏 담았다.
* **선물/상황 중심:** **사과**와 **바나나**가 들어 있는 **과일 바구니**를 선물로 받았습니다.


## 3. Step 배열

`store=False` 상태에서 대화 이력을 직접 관리할 때 사용합니다. 첫 요청의 모델 Step을 그대로 보관한 뒤 새로운 `user_input` Step을 추가합니다. 모델이 반환한 Step에는 서명 등 필요한 정보가 포함될 수 있으므로 임의로 다시 만들지 않고 그대로 전달합니다.

In [ ]:
history = [
    {
        "type": "user_input",
        "content": [
            {"type": "text", "text": "내 이름은 민수야."}
        ],
    }
]

first_turn = client.interactions.create(
    model=model,
    input=history,
    store=False,
)
print("첫 번째 답변:", first_turn.output_text)

history.extend(step.model_dump() for step in first_turn.steps)
history.append(
    {
        "type": "user_input",
        "content": [
            {"type": "text", "text": "내 이름이 뭐였지?"}
        ],
    }
)

second_turn = client.interactions.create(
    model=model,
    input=history,
    store=False,
)
print("두 번째 답변:", second_turn.output_text)

## 정리

- 간단한 질문은 문자열로 전달합니다.
- 한 번의 요청에 여러 내용을 넣을 때는 Content 배열을 사용합니다.
- 대화 이력을 직접 관리할 때는 Step 배열을 사용합니다.
- `user_input` 안의 `content`에 `text`, `image` 같은 Content가 들어갑니다.